In [1]:
# https://wikidocs.net/63618
import torch # cost 함수만 정의하면 파라미터에 대한 코스트의 기울기를 자동으로 계산해주는 모듈
import torchvision.datasets as datasets # datasets - 여러 데이터셋 포함
import torchvision.transforms as transforms # transforms - mnist가져올 때 변형해주는 모듈
import torch.nn.init # nn - 뉴럴 네트워크
from time import time # 코드 실행 시간 
import numpy as np

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu' # gpu가 더 효과적임. gpu쓸 때 사용하는 코드
#device = 'cpu' 
print(device)

# 랜덤 시드 고정
torch.manual_seed(777)

# GPU 사용 가능일 경우 랜덤 시드 고정
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

cuda


In [3]:
learning_rate = 0.001
training_epochs = 15 # 6만장을 다 돌면 한 epoch
batch_size = 100

In [4]:
train_dataset = datasets.MNIST(root='MNIST/', # 다운로드 경로 지정
                          train=True, # True를 지정하면 훈련 데이터로 다운로드
                          transform=transforms.ToTensor(), # 텐서로 변환 -> 기울기 계산을 위함
                          download=True)

test_dataset = datasets.MNIST(root='MNIST/', # 다운로드 경로 지정
                         train=False, # False를 지정하면 테스트 데이터로 다운로드
                         transform=transforms.ToTensor(), # 텐서로 변환
                         download=True)

Using downloaded and verified file: MNIST/MNIST\raw\train-images-idx3-ubyte.gz
Extracting MNIST/MNIST\raw\train-images-idx3-ubyte.gz to MNIST/MNIST\raw
Using downloaded and verified file: MNIST/MNIST\raw\train-labels-idx1-ubyte.gz
Extracting MNIST/MNIST\raw\train-labels-idx1-ubyte.gz to MNIST/MNIST\raw
Using downloaded and verified file: MNIST/MNIST\raw\t10k-images-idx3-ubyte.gz
Extracting MNIST/MNIST\raw\t10k-images-idx3-ubyte.gz to MNIST/MNIST\raw
Using downloaded and verified file: MNIST/MNIST\raw\t10k-labels-idx1-ubyte.gz
Extracting MNIST/MNIST\raw\t10k-labels-idx1-ubyte.gz to MNIST/MNIST\raw
Processing...
Done!


C:\Users\10015793\Miniconda3\lib\site-packages\torchvision\datasets\mnist.py:480: UserWarning: The given NumPy array is not writeable, and PyTorch does not support non-writeable tensors. This means you can write to the underlying (supposedly non-writeable) NumPy array using the tensor. You may want to copy the array to protect its data or make it writeable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at  ..\torch\csrc\utils\tensor_numpy.cpp:141.)
  return torch.from_numpy(parsed.astype(m[2], copy=False)).view(*s)


In [5]:
# batch 사이즈만큼 데이터를 가져오는 것을 유용하게 해주는 코드
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, # mnist_train에서 가져옴
                                          batch_size=batch_size, # 얼만큼?
                                          shuffle=True, # 순서대로 가져오지 말고 임의로 가져옴
                                          drop_last=True) #drop_last는 짜투리 처리. 가령 650 train data에서 batch_size = 100이었다면 마지막에 남은 50개의 데이터는 무시하는 것이 drop_last = True이고, 마지막 batch로는 그냥 50개의 데이터만 사용하는 것이 drop_last = False이다.

In [6]:
len(train_dataset), len(test_dataset)

(60000, 10000)

In [7]:
for (X_train, y_train) in train_loader:
    print('X_train:', X_train.size(), 'type:', X_train.type())
    print('y_train:', y_train.size(), 'type:', y_train.type())
    break

X_train: torch.Size([100, 1, 28, 28]) type: torch.FloatTensor
y_train: torch.Size([100]) type: torch.LongTensor


In [8]:
class CNN(torch.nn.Module): # 뉴럴네트워크 모듈을 상속받아서 만드는 클래스

    def __init__(self):
        super(CNN, self).__init__() # 뉴럴네트워크 모듈에 있는 모든 초기 과정을 상속
        self.keep_prob = 0.5

        # 첫번째 레이어
        # L1 ImgIn shape=(?, 28, 28, 1)
        #    Conv     -> (?, 28, 28, 32)
        #    Pool     -> (?, 14, 14, 32)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1), # convolution 필터 32개(이미지 하나 당 32개), 3x3 커널 -> S를 구함
            torch.nn.ReLU(), # 활성화함수
            torch.nn.MaxPool2d(kernel_size=2, stride=2)) # 2x2로 max pooling

        # 32개를 두번째 레이어에 입력
        # L2 ImgIn shape=(?, 14, 14, 32)
        #    Conv      ->(?, 14, 14, 64)
        #    Pool      ->(?, 7, 7, 64)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))
        
        # 64개를 세번째 레이어에 입력
        # L3 ImgIn shape=(?, 7, 7, 64)
        #    Conv      ->(?, 7, 7, 128)
        #    Pool      ->(?, 4, 4, 128)
        self.layer3 = torch.nn.Sequential(
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=1))

        # L4 FC 4x4x128 inputs -> 625 outputs (1차원으로 펼침)
        self.fc1 = torch.nn.Linear(4 * 4 * 128, 625, bias=True)
        torch.nn.init.xavier_uniform_(self.fc1.weight) # weight를 new weighting하는 과정
        self.layer4 = torch.nn.Sequential(
            self.fc1,
            torch.nn.ReLU(),
            torch.nn.Dropout(p=1 - self.keep_prob)) # self.keep_prob=0.5 / weight가 작은 것들은 가지치기 (0으로 만들어버림) -> 오버피팅 방지
        # L5 Final FC 625 inputs -> 10 outputs 10개의 출력과 연결 (원핫인코딩을 쓰기 때문)
        self.fc2 = torch.nn.Linear(625, 10, bias=True)
        torch.nn.init.xavier_uniform_(self.fc2.weight) # weight를 new weighting하는 과정

    def forward(self, x):
        out = self.layer1(x) # convoluted layer1
        out = self.layer2(out) # convoluted layer2
        out = self.layer3(out) # convoluted layer3
        out = out.view(out.size(0), -1)   # Flatten them for FC
        out = self.layer4(out) # fully connected layer
        out = self.fc2(out)
        return out

In [9]:
# CNN 모델 정의
model = CNN().to(device) # gpu에 뉴럴네트워크 structure를 넣어줌

In [10]:
# cost 함수
criterion = torch.nn.CrossEntropyLoss().to(device) # 비용 함수에 소프트맥스 함수 포함되어져 있음. z와 ztrue 차이를 정량화한 것
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # 경사하강법 쓸 때 optimizer를 사용

In [11]:
total_batch = len(train_loader)
print('총 배치의 수 : {}'.format(total_batch)) # batch_size = 100이니까 6만/100 = 600

총 배치의 수 : 600


In [12]:
time0 = time() # 코드 실행시간 측정 (현재시간)
for epoch in range(training_epochs): # 15개 epoch만큼 돎
    avg_cost = 0 

    for X, Y in train_loader: # 600개 중에서 미니 배치 단위로 꺼내온다. X는 미니 배치, Y는 레이블.
        # image is already size of (28x28), no reshape
        # label is not one-hot encoded
        # 입출력값은 device에 올림
        X = X.to(device) 
        Y = Y.to(device)

        optimizer.zero_grad() # 처음엔 기울기가 0
        hypothesis = model(X) # 순전파 (레이어로 값을 넘기고 넘기고 넘겨서 z값을 얻는 과정)
        cost = criterion(hypothesis, Y) # z와 ztrue 비교
        cost.backward() # 역전파
        optimizer.step()

        avg_cost += cost / total_batch

    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))
    
print("\nTraining Time (in minutes) =",(time()-time0)/60) # 종료 시간. 60은 분단위로 표현하기 위함

[Epoch:    1] cost = 0.187006295
[Epoch:    2] cost = 0.051727172
[Epoch:    3] cost = 0.0388577841
[Epoch:    4] cost = 0.0300376602
[Epoch:    5] cost = 0.0242262315
[Epoch:    6] cost = 0.020544
[Epoch:    7] cost = 0.0158695504
[Epoch:    8] cost = 0.0136439642
[Epoch:    9] cost = 0.0130596645
[Epoch:   10] cost = 0.0110945227
[Epoch:   11] cost = 0.00962128211
[Epoch:   12] cost = 0.00828585774
[Epoch:   13] cost = 0.00924284197
[Epoch:   14] cost = 0.00832668878
[Epoch:   15] cost = 0.00569887878

Training Time (in minutes) = 1.5271071195602417


In [13]:
# test
with torch.no_grad(): # 토치는 기울기를 계속 달고다니면서 계산하는데, 학습(기울기 계산)을 진행하지 않을 것이므로 torch.no_grad()
    X_test = test_dataset.test_data.view(len(test_dataset), 1, 28, 28).float().to(device)
    Y_test = test_dataset.test_labels.to(device)

    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction, 1) == Y_test # 원핫인코딩에서 제일 큰 값이 라벨
    accuracy = correct_prediction.float().mean())
    print('Accuracy:', accuracy.item())

C:\Users\10015793\Miniconda3\lib\site-packages\torchvision\datasets\mnist.py:63: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
C:\Users\10015793\Miniconda3\lib\site-packages\torchvision\datasets\mnist.py:53: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


Accuracy: 0.9784999489784241


In [14]:
print((correct_prediction))
print((correct_prediction.float()))
print((correct_prediction.float().mean()))

tensor([True, True, True,  ..., True, True, True], device='cuda:0')
tensor([1., 1., 1.,  ..., 1., 1., 1.], device='cuda:0')
tensor(0.9785, device='cuda:0')
